In [1]:
import numpy as np
import pandas as pd
import urllib
import json
from pathlib import Path

/cvmfs/cdms.opensciencegrid.org/releases/centos7/V04-06/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)


## Import metadata from Google Sheets

* [Access Google Sheet](https://docs.google.com/spreadsheets/d/1YklUMHrcqeEHhJtmiXgvCnb2B_SaIt99sZ-T6u1Kszc/edit#gid=0)

In [2]:
sheet_id = "1YklUMHrcqeEHhJtmiXgvCnb2B_SaIt99sZ-T6u1Kszc"
sheet_name = urllib.parse.quote("Raw data v2")
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)

In [3]:
# remove empty columns

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [4]:
df.head()

,Facility,nFridgeRun,DCRCversion,DCRCFirmwareVersion,Series,nDataType,SourcePos_Cs,SourcePos_Ba,Category,Filetype,AnalysisTag,HV,Shield,WorkingPoint,MCTemperature,Path
0,25,13,RevE_SN19,a58c184,25220121_160513,noSource,NaN,NaN,Raw,.mid.gz,TRnoise,0,open,43/43/25/25/18/14/39/36,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
1,25,13,RevE_SN19,a58c184,25220121_161311,noSource,NaN,NaN,Raw,.mid.gz,SCnoise,0,open,0/0/0/0/0/0,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
2,25,13,RevE_SN19,a58c184,25220121_161617,noSource,NaN,NaN,Raw,.mid.gz,NRnoise,0,open,500/500/500/500/500/500,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
3,25,13,RevE_SN19,a58c184,25220121_172740,noSource,NaN,NaN,Raw,.mid.gz,100V_rate,100,open,43/43/25/25/18/14/39/39,16.0,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
4,25,13,RevE_SN19,a58c184,25220122_164022,noSource,NaN,NaN,Raw,.mid.gz,0V_rate,0,open,43/43/25/25/18/14/39/39,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw


In [5]:
df.columns

Index(['Facility', 'nFridgeRun', 'DCRCversion', 'DCRCFirmwareVersion',
       'Series', 'nDataType', 'SourcePos_Cs', 'SourcePos_Ba', 'Category',
       'Filetype', 'AnalysisTag', 'HV', 'Shield', 'WorkingPoint',
       'MCTemperature', 'Path'],
      dtype='object')

In [6]:
!head NEXUS_R13_unblinded_raw_files.txt

/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220405_152020/25220405_152020_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220405_154619/25220405_154619_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220405_155041/25220405_155041_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220406_094740/25220406_094740_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220406_094227/25220406_094227_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220406_094148/25220406_094148_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220406_094109/25220406_094109_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220405_155351/25220405_155351_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220405_151740/25220405_151740_F0001.mid.gz
/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220329_143754/25220329_143754_F1190.mid.gz


## Data and functions used for conversion

In [7]:
facilities = {25: "NEXUS"}

In [8]:
## From previous analysis
n_data_types = {
    "Test": -1,
    "Background": 0,
    "Co": 1,
    "Co LowR": 2,
    "Cf": 3,
    "Rand": 4,
    "Mon": 7,
    "Cs source": 8,
    "Ba": 9,
    "YBe": 12,
    "SbBe": 13,
    "Y Blank": 14,
    "Sb Blank": 15,
    "Laser": 16,
    "Beam": 17,
    "Beam + Laser": 18,
    "Fe55 source": 19,
    "Fe55": 19,
    "Co57": 20,
    "IV curve": 100,
    "dIdV": 101,
    "NM noise": 102,
    "SC noise": 103,
    "TS noise": 104,
}

## New for Nexus 13

n_data_types["Cs-137"] = 8 # same as "Cs source"
n_data_types["noSource"] = 0 # like "Low Background"

In [9]:
def extract_nDump(filename):
    return int(filename.split("/")[-1].split("F")[1].split(".")[0])

In [10]:
assert extract_nDump('/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220319_095807/25220319_095807_F1200.mid.gz') == 1200

## Only register unblinded data

In [11]:
len(df)

127

In [12]:
df = df[df["Path"].str.contains("unblinded")]

In [13]:
len(df)

17

## List file in series

In [14]:
file_list = pd.read_table("NEXUS_R13_unblinded_raw_files.txt", squeeze=True, header=None)

In [15]:
def list_file(path, series):
    """List files at path
    
    This works without actual access to the filesystem, reads from pregenerated text file"""
    return file_list[file_list.str.contains("/".join([path, series]))].to_list()

## Iterate by row and register the data

* Iterate through all rows in the Pandas Dataframe
* Build a dictionary of processed metadata
* Call `ContinuousRawData` with the `record` dictionary as arguments

In [16]:
records = []
for row in df.to_dict(orient="records"):
    record = dict(
        filePath=row.pop("Path"),
        facility=facilities[row.pop("Facility")],
        nFridgeRun=row.pop("nFridgeRun"),
        nDataType=n_data_types.get(row.pop("nDataType"), np.nan),
        Series=row.pop("Series"),
        nIsJunk=0,
    )
    record["commentStart"] = [json.dumps(row)]
    for file in list_file(record["filePath"], record["Series"]):
        record["fileName"] = file
        record["nDumpNum"] = extract_nDump(file)
        records.append(record)

In [17]:
len(records)

6708

In [18]:
pd.DataFrame(records).to_csv("Nexus_13_registration_unblinded.csv")